# Hamiltonian Mechanics and Phase Space

**MechanicsDSL Notebook 05** | Classical Mechanics | Advanced

The Hamiltonian formulation recasts mechanics in terms of generalised coordinates $q_i$ and conjugate momenta $p_i = \\partial L/\\partial \\dot{q}_i$. This notebook explores the phase space structure of the pendulum, demonstrates the symplectic geometry of Hamiltonian flow, and compares standard RK4 with a symplectic integrator.

**MechanicsDSL automatically:**
- Performs the Legendre transform $H = \\sum_i p_i\\dot{q}_i - L$
- Derives Hamilton's equations: $\\dot{q}_i = \\partial H/\\partial p_i$, $\\dot{p}_i = -\\partial H/\\partial q_i$
- Identifies the symplectic structure for symplectic integrator selection

**DSL specification:**
```
\\system{pendulum_hamiltonian}
\\parameter{m}{1.0}{kg}
\\parameter{l}{1.0}{m}
\\lagrangian{0.5*m*l^2*\\dot{theta}^2 - m*g*l*(1-cos(theta))}
\\target{python_numpy}
```

MechanicsDSL derives: $H = p_\\theta^2/(2ml^2) + mgl(1-\\cos\\theta)$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

In [ ]:
m, l, g = 1.0, 1.0, 9.81

# MechanicsDSL-generated Hamilton's equations
# H = p^2/(2*m*l^2) + m*g*l*(1-cos(theta))
# dtheta/dt = dH/dp   = p/(m*l^2)
# dp/dt     = -dH/dtheta = -m*g*l*sin(theta)

def hamiltons_eqs(t, y):
    theta, p = y
    return [p/(m*l**2), -m*g*l*np.sin(theta)]

def hamiltonian(theta, p):
    return p**2/(2*m*l**2) + m*g*l*(1-np.cos(theta))

# Symplectic Euler integrator (preserves phase space volume)
def symplectic_euler_step(theta, p, dt):
    p_new     = p     - dt * m*g*l*np.sin(theta)  # update momentum first
    theta_new = theta + dt * p_new/(m*l**2)        # then position
    return theta_new, p_new

print('Hamilton equations and integrators defined.')

## Phase Space Structure

Phase space trajectories are level sets of the Hamiltonian $H(\\theta, p) = E$. The topology changes at the separatrix ($E = 2mgl$):
- $E < 2mgl$: closed curves (libration)
- $E = 2mgl$: separatrix (unstable equilibrium)
- $E > 2mgl$: open curves (rotation)

In [ ]:
fig, ax = plt.subplots(figsize=(11,7))
theta_grid = np.linspace(-np.pi, np.pi, 400)

# Energy levels
E_separatrix = 2*m*g*l
energy_levels = np.linspace(0.05*E_separatrix, 2.5*E_separatrix, 16)

for E in energy_levels:
    p2 = 2*m*l**2*(E - m*g*l*(1-np.cos(theta_grid)))
    mask = p2 >= 0
    if mask.any():
        p_pos = np.where(mask, np.sqrt(np.maximum(p2,0)), np.nan)
        color = 'tab:blue' if E < E_separatrix else 'tab:orange'
        alpha = 0.6
        ax.plot(theta_grid, p_pos,  color=color, lw=0.8, alpha=alpha)
        ax.plot(theta_grid, -p_pos, color=color, lw=0.8, alpha=alpha)

# Separatrix
p2_sep = 2*m*l**2*(E_separatrix - m*g*l*(1-np.cos(theta_grid)))
p_sep = np.where(p2_sep>=0, np.sqrt(np.maximum(p2_sep,0)), np.nan)
ax.plot(theta_grid,  p_sep, 'r-', lw=2.0, label='Separatrix ($E=2mgl$)')
ax.plot(theta_grid, -p_sep, 'r-', lw=2.0)

ax.set_xlabel(r'$\theta$ (rad)'); ax.set_ylabel(r'$p_\theta$ (kg⋅m²/s)')
ax.set_title('Pendulum Phase Space — Hamiltonian Level Sets')
ax.legend(); ax.set_xlim(-np.pi, np.pi)
ax.text(0, 0, 'Libration', ha='center', color='tab:blue', fontsize=10)
ax.text(2.5, 8, 'Rotation', ha='center', color='tab:orange', fontsize=10)
plt.tight_layout(); plt.show()

## Symplectic vs Standard Integration

Standard RK4 does not preserve the symplectic structure of Hamiltonian flow. Over long times it introduces secular energy drift. Symplectic integrators preserve this structure exactly, giving bounded (oscillatory) energy error for all time.

In [ ]:
dt = 0.02
t_end = 500.0
n_steps = int(t_end/dt)
theta0, p0 = 0.3, 0.0

# RK4
sol_rk4 = solve_ivp(hamiltons_eqs, [0, t_end], [theta0, p0],
                    max_step=dt, method='RK45', rtol=1e-6, atol=1e-8)
E_rk4 = hamiltonian(sol_rk4.y[0], sol_rk4.y[1])

# Symplectic Euler
theta_s, p_s = theta0, p0
t_arr = np.arange(0, t_end, dt)
E_sym = np.zeros(n_steps)
for i in range(n_steps):
    theta_s, p_s = symplectic_euler_step(theta_s, p_s, dt)
    E_sym[i] = hamiltonian(theta_s, p_s)

E0 = hamiltonian(theta0, p0)
drift_rk4 = np.abs((E_rk4 - E0)/E0)
drift_sym  = np.abs((E_sym  - E0)/E0)

fig, ax = plt.subplots(figsize=(11,4))
ax.semilogy(sol_rk4.t, drift_rk4, lw=0.6, label='RK45 (non-symplectic)', alpha=0.8)
ax.semilogy(t_arr,     drift_sym,  lw=0.6, label='Symplectic Euler', alpha=0.8)
ax.set_xlabel('Time (s)'); ax.set_ylabel(r'$|\Delta E/E_0|$')
ax.set_title(f'Energy drift over {t_end:.0f} s: symplectic vs standard integrator')
ax.legend(); plt.tight_layout(); plt.show()
print(f'RK45 max drift: {drift_rk4.max():.2e}')
print(f'Symplectic Euler max drift: {drift_sym.max():.2e}  (bounded, not secular)')

## Summary
| Property | Value |
|----------|-------|
| Hamiltonian | $H = p_\\theta^2/(2ml^2) + mgl(1-\\cos\\theta)$ |
| Separatrix energy | $E_s = 2mgl$ |
| Below separatrix | Libration (closed orbits) |
| Above separatrix | Rotation (open orbits) |
| Symplectic integrator drift | Bounded (oscillatory) |
| RK4 drift | Secular (grows with time) |

MechanicsDSL selects symplectic integrators automatically for Hamiltonian systems when `\\target{python_numpy}` is specified with `method: symplectic`.

**Next:** [06 — Rigid Body Dynamics](06_rigid_body.ipynb)